In [3]:
import torch                      # Core PyTorch library
import torch.nn as nn             # Neural network module
import torch.optim as optim       # Optimization algorithms


# ----------------------------------
# 1️⃣ Select Device
# ----------------------------------

# If GPU is available → use it
# otherwise use CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", device)

# -----------------------------
# 1️⃣ Create Toy Dataset
# -----------------------------

# Set random seed for reproducibility
torch.manual_seed(42)

# Number of samples
num_samples = 1000

# Sequence length (time steps)
seq_length = 5

# Each element has 1 feature
input_size = 1

# Generate random sequences
X = torch.randn(num_samples, seq_length, input_size).to(device)
# Label = 1 if sum > 0 else 0
y = (X.sum(dim=1) > 0).float().view(-1, 1).to(device)


# y shape should be (num_samples, 1)
# y = y.view(-1, 1)

# -----------------------------
# 2️⃣ Define Simple RNN Model
# -----------------------------

class SimpleRNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(SimpleRNN, self).__init__()

        # RNN layer
        # input_size = number of features per time step
        # hidden_size = number of hidden units
        self.rnn = nn.RNN(input_size, hidden_size, batch_first=True)

        # Fully connected layer for classification
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        # x shape: (batch_size, seq_length, input_size)

        # Pass through RNN
        # out shape: (batch_size, seq_length, hidden_size)
        # hidden shape: (1, batch_size, hidden_size)
        out, hidden = self.rnn(x)

        # We only use the LAST time step output
        # out[:, -1, :] means:
        #   take all batches
        #   last time step
        #   all hidden features
        last_output = out[:, -1, :]

        # Pass through fully connected layer
        output = self.fc(last_output)

        return output

# -----------------------------
# 3️⃣ Initialize Model
# -----------------------------

hidden_size = 8
output_size = 1

model = SimpleRNN(input_size, hidden_size, output_size).to(device)

# Binary classification → use BCEWithLogitsLoss
criterion = nn.BCEWithLogitsLoss()

# Adam optimizer
optimizer = optim.Adam(model.parameters(), lr=0.01)

# -----------------------------
# 4️⃣ Training Loop
# -----------------------------

num_epochs = 20

for epoch in range(num_epochs):

    # Zero gradients from previous step
    optimizer.zero_grad()

    # Forward pass
    outputs = model(X)

    # Compute loss
    loss = criterion(outputs, y)

    # Backpropagation
    loss.backward()

    # Update weights
    optimizer.step()

    # Print loss every 5 epochs
    if (epoch+1) % 5 == 0:
        print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}")

# -----------------------------
# 5️⃣ Inference (Prediction)
# -----------------------------

# Switch to evaluation mode
model.eval()

# Disable gradient calculation (faster inference)
with torch.no_grad():

    # Take a new random sequence
    test_sample = torch.randn(1, seq_length, input_size).to(device)

    # Get model output
    raw_output = model(test_sample)

    # Apply sigmoid to convert logits → probability
    probability = torch.sigmoid(raw_output)

    # If probability > 0.5 → class 1 else 0
    prediction = (probability > 0.5).int()

    print("\nTest sequence:", test_sample)
    print("Predicted probability:", probability.item())
    print("Predicted class:", prediction.item())

Using device: cuda
Epoch [5/20], Loss: 0.6469
Epoch [10/20], Loss: 0.5744
Epoch [15/20], Loss: 0.4443
Epoch [20/20], Loss: 0.3394

Test sequence: tensor([[[-0.5777],
         [-0.4078],
         [-0.2749],
         [ 1.2669],
         [ 0.9620]]], device='cuda:0')
Predicted probability: 0.3887461721897125
Predicted class: 0


In [4]:
import torch
import torch.nn as nn
import torch.optim as optim

# ----------------------------------
# 1️⃣ Device (CPU or GPU)
# ----------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ----------------------------------
# 2️⃣ Create Toy Dataset
# ----------------------------------

torch.manual_seed(42)

num_samples = 1000
seq_length = 5
input_size = 1

# Random sequences
X = torch.randn(num_samples, seq_length, input_size).to(device)

# Label: 1 if sum > 0 else 0
y = (X.sum(dim=1) > 0).float().view(-1, 1).to(device)

# ----------------------------------
# 3️⃣ Define LSTM Model
# ----------------------------------

class SimpleLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(SimpleLSTM, self).__init__()

        # LSTM layer
        # input_size: features per time step
        # hidden_size: size of hidden state
        # batch_first=True → input shape = (batch, seq, features)
        self.lstm = nn.LSTM(input_size, hidden_size, batch_first=True)

        # Fully connected layer
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):

        # x shape: (batch_size, seq_length, input_size)

        # LSTM returns:
        # out: (batch_size, seq_length, hidden_size)
        # (h_n, c_n):
        #   h_n: final hidden state
        #   c_n: final cell state
        out, (h_n, c_n) = self.lstm(x)

        # We use last time step output
        last_output = out[:, -1, :]

        # Pass through fully connected layer
        output = self.fc(last_output)

        return output

# ----------------------------------
# 4️⃣ Initialize Model
# ----------------------------------

hidden_size = 8
output_size = 1

model = SimpleLSTM(input_size, hidden_size, output_size).to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

# ----------------------------------
# 5️⃣ Training Loop
# ----------------------------------

num_epochs = 20

for epoch in range(num_epochs):

    model.train()

    optimizer.zero_grad()

    outputs = model(X)

    loss = criterion(outputs, y)

    loss.backward()

    optimizer.step()

    if (epoch+1) % 5 == 0:
        print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}")

# ----------------------------------
# 6️⃣ Inference
# ----------------------------------

model.eval()

with torch.no_grad():

    test_sample = torch.randn(1, seq_length, input_size).to(device)

    raw_output = model(test_sample)

    probability = torch.sigmoid(raw_output)

    prediction = (probability > 0.5).int()

    print("\nTest sequence:", test_sample)
    print("Predicted probability:", probability.item())
    print("Predicted class:", prediction.item())

Using device: cuda
Epoch [5/20], Loss: 0.6924
Epoch [10/20], Loss: 0.6676
Epoch [15/20], Loss: 0.6461
Epoch [20/20], Loss: 0.6137

Test sequence: tensor([[[-0.0657],
         [-0.9046],
         [-0.4095],
         [-0.3990],
         [-1.0448]]], device='cuda:0')
Predicted probability: 0.45531773567199707
Predicted class: 0


In [5]:
import torch
import torch.nn as nn
import torch.optim as optim

# ----------------------------------
# 1️⃣ Device
# ----------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ----------------------------------
# 2️⃣ Toy Dataset
# ----------------------------------

torch.manual_seed(42)

num_samples = 1000
seq_length = 5
input_size = 1

X = torch.randn(num_samples, seq_length, input_size).to(device)
y = (X.sum(dim=1) > 0).float().view(-1, 1).to(device)

# ----------------------------------
# 3️⃣ Define Bidirectional RNN
# ----------------------------------

class BiRNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(BiRNN, self).__init__()

        # bidirectional=True → two RNNs internally
        self.rnn = nn.RNN(
            input_size,
            hidden_size,
            batch_first=True,
            bidirectional=True
        )

        # Because bidirectional → hidden_size * 2
        self.fc = nn.Linear(hidden_size * 2, output_size)

    def forward(self, x):

        # x shape: (batch_size, seq_length, input_size)

        # out shape:
        # (batch_size, seq_length, hidden_size * 2)
        out, hidden = self.rnn(x)

        # hidden shape:
        # (num_layers * 2, batch_size, hidden_size)

        # Take last time step output
        last_output = out[:, -1, :]

        # Pass through fully connected layer
        output = self.fc(last_output)

        return output

# ----------------------------------
# 4️⃣ Initialize Model
# ----------------------------------

hidden_size = 8
output_size = 1

model = BiRNN(input_size, hidden_size, output_size).to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

# ----------------------------------
# 5️⃣ Training
# ----------------------------------

num_epochs = 20

for epoch in range(num_epochs):

    model.train()

    optimizer.zero_grad()

    outputs = model(X)

    loss = criterion(outputs, y)

    loss.backward()

    optimizer.step()

    if (epoch+1) % 5 == 0:
        print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}")

# ----------------------------------
# 6️⃣ Inference
# ----------------------------------

model.eval()

with torch.no_grad():

    test_sample = torch.randn(1, seq_length, input_size).to(device)

    raw_output = model(test_sample)

    probability = torch.sigmoid(raw_output)

    prediction = (probability > 0.5).int()

    print("\nPredicted class:", prediction.item())

Using device: cuda
Epoch [5/20], Loss: 0.6665
Epoch [10/20], Loss: 0.6275
Epoch [15/20], Loss: 0.5425
Epoch [20/20], Loss: 0.4086

Predicted class: 1


In [6]:
import torch
import torch.nn as nn
import torch.optim as optim

# ----------------------------------
# 1️⃣ Device
# ----------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ----------------------------------
# 2️⃣ Toy Dataset
# ----------------------------------

torch.manual_seed(42)

num_samples = 1000
seq_length = 5
input_size = 1

# Random sequences
X = torch.randn(num_samples, seq_length, input_size).to(device)

# Label: 1 if sum > 0 else 0
y = (X.sum(dim=1) > 0).float().view(-1, 1).to(device)

# ----------------------------------
# 3️⃣ Define Bidirectional LSTM
# ----------------------------------

class BiLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(BiLSTM, self).__init__()

        # bidirectional=True → forward + backward LSTM
        self.lstm = nn.LSTM(
            input_size,
            hidden_size,
            batch_first=True,
            bidirectional=True
        )

        # Because bidirectional → hidden_size * 2
        self.fc = nn.Linear(hidden_size * 2, output_size)

    def forward(self, x):
        # x shape: (batch_size, seq_length, input_size)

        # LSTM output:
        # out shape:
        # (batch_size, seq_length, hidden_size * 2)
        #
        # h_n shape:
        # (num_layers * 2, batch_size, hidden_size)
        #
        # c_n shape:
        # (num_layers * 2, batch_size, hidden_size)

        out, (h_n, c_n) = self.lstm(x)

        # Take last time step
        last_output = out[:, -1, :]

        # Classification layer
        output = self.fc(last_output)

        return output

# ----------------------------------
# 4️⃣ Initialize Model
# ----------------------------------

hidden_size = 8
output_size = 1

model = BiLSTM(input_size, hidden_size, output_size).to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

# ----------------------------------
# 5️⃣ Training
# ----------------------------------

num_epochs = 20

for epoch in range(num_epochs):

    model.train()
    optimizer.zero_grad()

    outputs = model(X)

    loss = criterion(outputs, y)

    loss.backward()
    optimizer.step()

    if (epoch + 1) % 5 == 0:
        print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}")

# ----------------------------------
# 6️⃣ Inference
# ----------------------------------

model.eval()

with torch.no_grad():
    test_sample = torch.randn(1, seq_length, input_size).to(device)

    raw_output = model(test_sample)

    probability = torch.sigmoid(raw_output)

    prediction = (probability > 0.5).int()

    print("\nPredicted class:", prediction.item())

Using device: cuda
Epoch [5/20], Loss: 0.6675
Epoch [10/20], Loss: 0.6325
Epoch [15/20], Loss: 0.5759
Epoch [20/20], Loss: 0.4883

Predicted class: 0


In [7]:
import torch
import torch.nn as nn
import torch.optim as optim

# ----------------------------------
# 1️⃣ Device
# ----------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ----------------------------------
# 2️⃣ Toy Dataset
# ----------------------------------

torch.manual_seed(42)

num_samples = 1000
seq_length = 5
input_size = 1

X = torch.randn(num_samples, seq_length, input_size).to(device)
y = (X.sum(dim=1) > 0).float().view(-1, 1).to(device)

# ----------------------------------
# 3️⃣ Define Bidirectional GRU
# ----------------------------------

class BiGRU(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(BiGRU, self).__init__()

        # bidirectional=True creates forward + backward GRU
        self.gru = nn.GRU(
            input_size,
            hidden_size,
            batch_first=True,
            bidirectional=True
        )

        # Output dimension doubles because of bidirection
        self.fc = nn.Linear(hidden_size * 2, output_size)

    def forward(self, x):
        # x shape: (batch_size, seq_length, input_size)

        # out shape:
        # (batch_size, seq_length, hidden_size * 2)
        #
        # h_n shape:
        # (num_layers * 2, batch_size, hidden_size)

        out, h_n = self.gru(x)

        # Cleaner way: use final hidden states
        forward_last = h_n[0]      # forward direction
        backward_last = h_n[1]     # backward direction

        # Concatenate
        combined = torch.cat((forward_last, backward_last), dim=1)

        # Classification
        output = self.fc(combined)

        return output

# ----------------------------------
# 4️⃣ Initialize Model
# ----------------------------------

hidden_size = 8
output_size = 1

model = BiGRU(input_size, hidden_size, output_size).to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

# ----------------------------------
# 5️⃣ Training
# ----------------------------------

num_epochs = 20

for epoch in range(num_epochs):

    model.train()
    optimizer.zero_grad()

    outputs = model(X)

    loss = criterion(outputs, y)

    loss.backward()
    optimizer.step()

    if (epoch + 1) % 5 == 0:
        print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}")

# ----------------------------------
# 6️⃣ Inference
# ----------------------------------

model.eval()

with torch.no_grad():

    test_sample = torch.randn(1, seq_length, input_size).to(device)

    raw_output = model(test_sample)

    probability = torch.sigmoid(raw_output)

    prediction = (probability > 0.5).int()

    print("\nPredicted class:", prediction.item())

Using device: cuda
Epoch [5/20], Loss: 0.6721
Epoch [10/20], Loss: 0.6292
Epoch [15/20], Loss: 0.5773
Epoch [20/20], Loss: 0.5087

Predicted class: 0
